In [ ]:
# Install PySpark and ML libraries
!pip install pyspark findspark scikit-learn pandas matplotlib seaborn

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HIGGS Big Data ML Project") \
    .config("spark.executor.memory","4g") \
    .config("spark.driver.memory","4g") \
    .config("spark.sql.shuffle.partitions","200") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.2


In [ ]:
pip install findspark

In [ ]:
!wget https://archive.ics.uci.edu/ml/machine-learning-databases/00280/HIGGS.csv.gz

--2026-02-25 14:17:42--  https://archive.ics.uci.edu/ml/machine-learning-databases/00280/HIGGS.csv.gz
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘HIGGS.csv.gz’

HIGGS.csv.gz            [              <=>   ]   2.62G  32.8MB/s    in 2m 9s   

2026-02-25 14:19:52 (20.7 MB/s) - ‘HIGGS.csv.gz’ saved [2816407858]



In [ ]:
df = spark.read.csv(
    "/content/HIGGS.csv.gz",
    header=False,
    inferSchema=True
)

print("Dataset shape:", df.count(), "rows,", len(df.columns), "columns")
df.show(5)

Dataset shape: 11000000 rows, 29 columns
+---+------------------+-------------------+-------------------+------------------+-------------------+------------------+--------------------+-------------------+------------------+------------------+-------------------+-------------------+------------------+------------------+--------------------+-------------------+-----------------+-------------------+--------------------+--------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|_c0|               _c1|                _c2|                _c3|               _c4|                _c5|               _c6|                 _c7|                _c8|               _c9|              _c10|               _c11|               _c12|              _c13|              _c14|                _c15|               _c16|             _c17|               _c18|                _c19|                _c20|       

In [ ]:
df.write.mode("overwrite").parquet("higgs_parquet")

df = spark.read.parquet("higgs_parquet")

In [ ]:
from pyspark.sql.functions import col

label_col = "_c0"

feature_cols = df.columns[1:]

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

data = assembler.transform(df).select(col(label_col).alias("label"),"features")

In [ ]:
train_data, test_data = data.randomSplit([0.8,0.2], seed=42)

train_data.cache()
test_data.cache()

DataFrame[label: double, features: vector]

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label")

lr_model = lr.fit(train_data)

lr_pred = lr_model.transform(test_data)

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label"
)

dt_model = dt.fit(train_data)

dt_pred = dt_model.transform(test_data)

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50
)

rf_model = rf.fit(train_data)

rf_pred = rf_model.transform(test_data)

In [ ]:
from pyspark.ml.classification import LinearSVC

svm = LinearSVC(
    featuresCol="features",
    labelCol="label"
)

svm_model = svm.fit(train_data)

svm_pred = svm_model.transform(test_data)

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator


evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)


print("Logistic Regression AUC:", evaluator.evaluate(lr_pred))
print("Decision Tree AUC:", evaluator.evaluate(dt_pred))
print("Random Forest AUC:", evaluator.evaluate(rf_pred))
print("SVM AUC:", evaluator.evaluate(svm_pred))

In [ ]:
sample_df = df.limit(100000).toPandas()

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X = sample_df.iloc[:,1:]
y = sample_df.iloc[:,0]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

model = RandomForestClassifier()

model.fit(X_train,y_train)

pred = model.predict_proba(X_test)[:,1]

print("Sklearn AUC:", roc_auc_score(y_test,pred))

Sklearn AUC: 0.7966718574548112


In [ ]:
import pandas as pd

importance = rf_model.featureImportances

feature_importance = pd.DataFrame({
    "feature":feature_cols,
    "importance":importance.toArray()
})

feature_importance.to_csv("feature_importance.csv",index=False)

In [ ]:
v